In [ ]:
from ultralytics import YOLO
import torchvision
import tqdm
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.analysis import (
    build_split_summary_and_class_counts,
    collect_bbox_size_data,
    collect_small_images,
)

from src.utils import (
    change_file_type,
    yolo_preprocess,
    yolo_to_xyxy_pixels,
    analyze_missed_objects,
    get_image_size,
    box_iou_raw,
    load_yolo_labels,
    box_iou_batch
)

In [ ]:
YOLO_PATH = ""

In [ ]:
split_dirs, summary_df, class_distributions = build_split_summary_and_class_counts("./visdrone")
bbox_size_rows, bbox_size_df = collect_bbox_size_data(split_dirs)
small_images = collect_small_images(bbox_size_rows, split_name="train")
images_paths = list(zip(small_images, [change_file_type(v) for v in small_images]))

In [ ]:
model = YOLO(YOLO_PATH)

iou_threshold = 0.1

total_tp = 0
total_gt = 0

saved_predictions = []

for sample in small_images[:1000]:
    label_path = change_file_type(sample)

    decoded_sample = torchvision.io.decode_image(sample)

    # Run inference
    results = model(
        yolo_preprocess(decoded_sample).unsqueeze(0),
        conf=0.2,
        imgsz=640,
        verbose=False
    )

    result = results[0]

    pred_boxes = result.boxes.xyxy.cpu()
    pred_classes = result.boxes.cls.cpu()
    pred_conf = result.boxes.conf.cpu()
    saved_predictions.append((pred_boxes.numpy(), pred_classes.numpy().astype(int), pred_conf.numpy()))

    # Image dimensions corresponding to the model result
    h, w = result.orig_shape

    # Ground truth
    gt_boxes, gt_classes = load_yolo_labels(
        label_path,
        h,
        w
    )

    total_gt += len(gt_boxes)

    # Predictions
    if result.boxes is None or len(result.boxes) == 0:
        continue


    if len(gt_boxes) == 0:
        continue

    # IoU between every prediction and GT
    ious = box_iou_batch(pred_boxes, gt_boxes)

    # Process highest-confidence predictions first
    order = torch.argsort(pred_conf, descending=True)

    matched_gt = set()

    for pred_idx in order:
        pred_idx = pred_idx.item()

        # Only GT boxes of the same class are eligible
        valid_gt = [
            j for j in range(len(gt_boxes))
            if j not in matched_gt
            and gt_classes[j] == pred_classes[pred_idx]
        ]

        if not valid_gt:
            continue

        candidate_ious = ious[pred_idx, valid_gt]
        best_pos = torch.argmax(candidate_ious).item()
        best_gt = valid_gt[best_pos]
        best_iou = candidate_ious[best_pos].item()

        if best_iou >= iou_threshold:
            matched_gt.add(best_gt)
            total_tp += 1

false_negatives = total_gt - total_tp

recall = (
    total_tp / total_gt
    if total_gt > 0
    else 0.0
)

print(f"TP:     {total_tp}")
print(f"FN:     {false_negatives}")
print(f"GT:     {total_gt}")
print(f"Recall: {recall:.4f}")

In [ ]:
missed, class_stats, size_stats, tb = analyze_missed_objects(
    images_paths[:1000],
    saved_predictions[:1000],
    iou_threshold=0.5,
    conf_threshold=0.2,
)

print("Missed objects:", len(missed))

print("\nMisses by class:")
for cls, count in class_stats.most_common():
    print(f"class {cls}: {count}")

print("\nMisses by size:")
for size, count in size_stats.items():
    print(f"{size}: {count}")

In [ ]:
def calculate_recall(image_paths, results, conf_threshold=0.25, iou_threshold=0.5):
    total_gt = 0
    detected_gt = 0

    for idx, (image_path, label_path) in tqdm.tqdm(enumerate(image_paths), total = len(image_paths)):
        result = results[idx]

        gt_boxes = []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                _, x, y, w, h = map(float, parts[:5])
                gt_boxes.append([x, y, w, h])

        total_gt += len(gt_boxes)

        if not gt_boxes or result is None:
            continue

        pred_boxes = results[idx][0]
        pred_conf = results[idx][1]

        pred_boxes = pred_boxes[pred_conf >= conf_threshold]

        matched = set()

        img_w, img_h = get_image_size(image_path)

        for gt in gt_boxes:
            best_iou = 0
            best_idx = -1

            for pred_idx, pred in enumerate(pred_boxes):
                if pred_idx in matched:
                    continue
                gt_box_xyxy = yolo_to_xyxy_pixels(gt, img_w, img_h)
                iou = box_iou_raw(torch.tensor(gt_box_xyxy), torch.tensor(pred))

                if iou > best_iou:
                    best_iou = iou
                    best_idx = pred_idx
            if best_iou >= iou_threshold:
                detected_gt += 1
                matched.add(best_idx)

    return detected_gt / total_gt if total_gt > 0 else 0.0

In [ ]:
thresholds = np.arange(0.1, 1.01, 0.3)

recalls = [
    calculate_recall(
        images_paths[:1000],
        saved_predictions[:1000],
        conf_threshold=0.5,
        iou_threshold=threshold,
    )
    for threshold in thresholds
]

plt.figure(figsize=(8, 5))
plt.plot(thresholds, recalls, marker="o")
plt.xlabel("Confidence threshold")
plt.ylabel("Recall")
plt.title("YOLO Recall vs Confidence Threshold")
plt.grid()
plt.show()